# ADS Homework 3 - Deep Learning (PyTorch)

This notebook follows `.cursor/rules/task_description_hw3.mdc` and the plan in `hw3_plan.md`.

**Datasets (Kaggle paths):**
- **Telco Customer Churn**: `/kaggle/input/telco-customer-churn-realistic-customer-feedback/telco_churn_with_all_feedback.csv`
- **Flowers-102**: `/kaggle/input/pytorch-challange-flower-dataset`
- **Jena Climate**: `/kaggle/input/jena-climate`

We use PyTorch for MLP, CNN, RNN, and Transformer experiments with systematic architecture changes.

**Author:** [Your Name]


In [ ]:
# Core imports and setup
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Subset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.backends.cudnn.deterministic = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


## 1) Dataset Paths
Confirm these paths match Kaggle mounts.


In [ ]:
# Update if your Kaggle paths differ
TELCO_PATH = Path("/kaggle/input/telco-customer-churn-realistic-customer-feedback/telco_churn_with_all_feedback.csv")
FLOWERS_ROOT = Path("/kaggle/input/pytorch-challange-flower-dataset")
JENA_ROOT = Path("/kaggle/input/jena-climate")

print("Telco exists:", TELCO_PATH.exists())
print("Flowers root exists:", FLOWERS_ROOT.exists())
print("Jena root exists:", JENA_ROOT.exists())


# Part 1: MLP on Telco Customer Churn (Tabular)

**Tasks:**
1. Binary Classification: Predict `Churn`
2. Regression: Predict `TotalCharges`

**Preprocessing:**
- Numeric: Median imputation + Standard Scaling
- Categorical: Most Frequent imputation + One-Hot Encoding


In [ ]:
def build_preprocessor(df, target_cols):
    # Identify feature columns
    feature_cols = [c for c in df.columns if c not in target_cols]
    X = df[feature_cols]

    numeric_cols = X.select_dtypes(include=["number"]).columns.tolist()
    categorical_cols = [c for c in X.columns if c not in numeric_cols]

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_cols),
            ("cat", categorical_transformer, categorical_cols),
        ]
    )

    return preprocessor

if TELCO_PATH.exists():
    # Load Telco dataset
    df_telco = pd.read_csv(TELCO_PATH)

    # Find churn column (case-insensitive)
    churn_col = next((c for c in df_telco.columns if c.lower() == "churn"), None)
    if churn_col is None:
        raise ValueError("Churn column not found.")

    # Regression target
    target_reg = "TotalCharges"
    if target_reg not in df_telco.columns:
        raise ValueError("TotalCharges column not found.")

    # Clean TotalCharges and fill NaNs with median
    df_telco[target_reg] = pd.to_numeric(df_telco[target_reg], errors="coerce")
    median_total = df_telco[target_reg].median()
    df_telco[target_reg] = df_telco[target_reg].fillna(median_total)

    # Binary target: Yes/No -> 1/0
    y_bin = df_telco[churn_col].apply(lambda x: 1 if str(x).lower() in ["yes", "true", "1"] else 0)

    # Remove ID columns from features if present
    id_cols = [c for c in df_telco.columns if c.lower() in ["customerid", "customer_id"]]

    # Prepare features for classification/regression
    feature_df = df_telco.drop(columns=[churn_col] + id_cols)
    preprocessor = build_preprocessor(feature_df, target_cols=[target_reg])

    # Classification features: drop targets
    X_cls = feature_df.drop(columns=[target_reg])

    # Fit preprocessor on classification features
    X_all = preprocessor.fit_transform(X_cls)
    # Guard against any residual NaNs from all-missing columns
    X_all = np.nan_to_num(X_all, nan=0.0, posinf=0.0, neginf=0.0)

    # Align with targets
    y_reg = df_telco[target_reg].values.astype(np.float32)
    y_reg = np.nan_to_num(y_reg, nan=np.nanmedian(y_reg), posinf=0.0, neginf=0.0)
    y_bin = y_bin.values.astype(np.float32)

    print("Feature matrix shape:", X_all.shape)

    # Train/val split
    X_train, X_test, y_bin_train, y_bin_test, y_reg_train, y_reg_test = train_test_split(
        X_all, y_bin, y_reg, test_size=0.2, random_state=SEED, stratify=y_bin
    )

    # Torch datasets
    train_ds_cls = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_bin_train, dtype=torch.float32))
    val_ds_cls = TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_bin_test, dtype=torch.float32))

    train_ds_reg = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_reg_train, dtype=torch.float32))
    val_ds_reg = TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_reg_test, dtype=torch.float32))
else:
    print("Warning: Telco dataset not found. Skipping Part 1 data loading.")


In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_sizes=(128, 64), dropout=0.2, activation="relu", use_batch_norm=False, out_dim=1):
        super().__init__()
        layers = []
        prev = input_dim
        
        # Activation selection
        if activation == "relu":
            Act = nn.ReLU
        elif activation == "leakyrelu":
            Act = nn.LeakyReLU
        elif activation == "tanh":
            Act = nn.Tanh
        elif activation == "sigmoid":
            Act = nn.Sigmoid
        else:
            Act = nn.ReLU

        for h in hidden_sizes:
            layers.append(nn.Linear(prev, h))
            if use_batch_norm:
                layers.append(nn.BatchNorm1d(h))
            layers.append(Act())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        
        layers.append(nn.Linear(prev, out_dim))
        self.net = nn.Sequential(*layers)
        
        # Weight initialization
        self._init_weights(activation)

    def _init_weights(self, activation):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                if activation in ["relu", "leakyrelu"]:
                    nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                elif activation in ["tanh", "sigmoid"]:
                    nn.init.xavier_normal_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def forward(self, x):
        return self.net(x)

def train_mlp(model, train_ds, val_ds, loss_fn, optimizer, epochs=20, batch_size=256):
    model.to(DEVICE)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    history = {"train_loss": [], "val_loss": []}

    start_time = time.time()
    for epoch in range(1, epochs + 1):
        model.train()
        train_losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            preds = model(xb).squeeze()
            loss = loss_fn(preds, yb)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                preds = model(xb).squeeze()
                loss = loss_fn(preds, yb)
                val_losses.append(loss.item())

        history["train_loss"].append(np.mean(train_losses))
        history["val_loss"].append(np.mean(val_losses))
        
        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch:02d} | Train Loss {history['train_loss'][-1]:.4f} | Val Loss {history['val_loss'][-1]:.4f}")
    
    print(f"Training finished in {time.time()-start_time:.2f}s")
    return history

def eval_classification(model, X, y_true):
    model.eval()
    with torch.no_grad():
        logits = model(torch.tensor(X, dtype=torch.float32).to(DEVICE)).squeeze().cpu().numpy()
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)
    return {
        "accuracy": accuracy_score(y_true, preds),
        "f1": f1_score(y_true, preds),
        "roc_auc": roc_auc_score(y_true, probs),
    }

def eval_regression(model, X, y_true):
    model.eval()
    with torch.no_grad():
        preds = model(torch.tensor(X, dtype=torch.float32).to(DEVICE)).squeeze().cpu().numpy()
    return {
        "mae": mean_absolute_error(y_true, preds),
        "mse": mean_squared_error(y_true, preds),
        "r2": r2_score(y_true, preds),
    }


### MLP Experiments
We will run 3 configurations for Classification to observe effects of optimization and architecture:
1. **Baseline**: Hidden=(128, 64), ReLU, Adam
2. **Experiment A (Optimization)**: Same structure, but SGD + Momentum
3. **Experiment B (Architecture)**: Deeper (256, 128, 64), Batch Norm, Dropout 0.3

In [ ]:
if TELCO_PATH.exists():
    input_dim = X_train.shape[1]
    
    print("--- Classification Experiments ---")
    # 1. Baseline
    print("\n1. Baseline MLP (Adam, ReLU)")
    model_base = MLP(input_dim, hidden_sizes=(128, 64), activation="relu")
    hist_base = train_mlp(model_base, train_ds_cls, val_ds_cls, nn.BCEWithLogitsLoss(), 
                          optim.Adam(model_base.parameters(), lr=1e-3), epochs=15)
    print("Metrics:", eval_classification(model_base, X_test, y_bin_test))

    # 2. Optimization: SGD + Momentum
    print("\n2. SGD + Momentum")
    model_sgd = MLP(input_dim, hidden_sizes=(128, 64), activation="relu")
    hist_sgd = train_mlp(model_sgd, train_ds_cls, val_ds_cls, nn.BCEWithLogitsLoss(), 
                         optim.SGD(model_sgd.parameters(), lr=0.01, momentum=0.9), epochs=15)
    print("Metrics:", eval_classification(model_sgd, X_test, y_bin_test))

    # 3. Architecture: Deeper + Batch Norm
    print("\n3. Deeper + BatchNorm")
    model_deep = MLP(input_dim, hidden_sizes=(256, 128, 64), activation="leakyrelu", 
                     use_batch_norm=True, dropout=0.3)
    hist_deep = train_mlp(model_deep, train_ds_cls, val_ds_cls, nn.BCEWithLogitsLoss(), 
                          optim.Adam(model_deep.parameters(), lr=1e-3), epochs=15)
    print("Metrics:", eval_classification(model_deep, X_test, y_bin_test))

    # Plot Losses
    plt.figure(figsize=(10, 5))
    plt.plot(hist_base['val_loss'], label='Baseline (Adam)')
    plt.plot(hist_sgd['val_loss'], label='SGD+Mom')
    plt.plot(hist_deep['val_loss'], label='Deep+BN')
    plt.title("MLP Classification Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("BCE Loss")
    plt.legend()
    plt.show()


### Discussion Question 1 (MLP)
*   **Why are neural networks so powerful?**
*   **Why does training become more difficult as we go deeper?**
*   *(Optional) Universal Approximation vs. Depth benefits.*

*(Double-click to edit and answer here)*


# Part 2: CNN on Flowers-102 (Image Classification)

**Task**: Multi-class classification (102 classes).

**Methods**:
1. **Custom CNN**: Experiments with Kernel size, Strides, Depth.
2. **Transfer Learning**: ResNet18 (Feature Extraction/Fine-tuning).

**Data Augmentation**: Random Crops, Flips, Rotations.


In [ ]:
import torchvision
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.models import resnet18, ResNet18_Weights

# 1. Setup Data with Augmentation
train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_tfms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Attempt to find data path structure
def get_flower_loaders(root_path, batch_size=32):
    if not root_path.exists():
        print("Flowers root not found.")
        return None, None, 0
    
    # Check for train/valid folders
    train_dir = root_path / 'train'
    val_dir = root_path / 'valid'
    
    if not train_dir.exists():
        # Fallback: single folder, split it
        print("Train folder not explicitly found, checking subdirs...")
        # (Simple logic: if just one folder with classes, use Subset)
        # Assuming standard Kaggle structure for this dataset often has 'train'/'valid'/'test'
        # If not, allow user to adjust path manually.
        return None, None, 0

    train_ds = ImageFolder(train_dir, transform=train_tfms)
    val_ds = ImageFolder(val_dir, transform=val_tfms)
    
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2)
    
    return train_loader, val_loader, len(train_ds.classes)

# If paths exist, create loaders
if FLOWERS_ROOT.exists():
    # Adjust if dataset is nested
    # (Logic from previous notebook kept short)
    train_loader_img, val_loader_img, num_classes = get_flower_loaders(FLOWERS_ROOT)
    if train_loader_img is None:
        print("Could not automatically load Flowers dataset structure. Please check paths.")
else:
    print("Flowers dataset not found.")


In [ ]:
class CustomCNN(nn.Module):
    def __init__(self, num_classes, base_channels=32, kernel_size=3, pool_type='max'):
        super().__init__()
        padding = kernel_size // 2
        
        def make_block(in_c, out_c):
            layers = [
                nn.Conv2d(in_c, out_c, kernel_size, padding=padding),
                nn.ReLU(),
                nn.BatchNorm2d(out_c)
            ]
            if pool_type == 'max':
                layers.append(nn.MaxPool2d(2))
            elif pool_type == 'avg':
                layers.append(nn.AvgPool2d(2))
            return layers

        self.features = nn.Sequential(
            *make_block(3, base_channels),
            *make_block(base_channels, base_channels*2),
            *make_block(base_channels*2, base_channels*4),
            *make_block(base_channels*4, base_channels*8),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.classifier = nn.Linear(base_channels*8, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = x.flatten(1)
        return self.classifier(x)

def train_cnn(model, train_loader, val_loader, epochs=5, lr=1e-3):
    if train_loader is None: return
    model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    for epoch in range(1, epochs+1):
        model.train()
        train_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
        # Validation
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                preds = model(xb).argmax(dim=1)
                correct += (preds == yb).sum().item()
                total += yb.size(0)
        
        print(f"Epoch {epoch} | Train Loss {train_loss/len(train_loader):.4f} | Val Acc {correct/total:.4f}")


### CNN Experiments
1. **Baseline**: Kernel=3, MaxPool, Base Channels=32
2. **Experiment**: Kernel=5 (Larger Receptive Field)
3. **Transfer Learning**: ResNet18 (Fine-tuning)

In [ ]:
if FLOWERS_ROOT.exists() and train_loader_img:
    print("--- CNN Experiments ---")
    
    print("\n1. Custom CNN (Kernel=3)")
    cnn_base = CustomCNN(num_classes, kernel_size=3)
    train_cnn(cnn_base, train_loader_img, val_loader_img, epochs=5)
    
    print("\n2. Custom CNN (Kernel=5)")
    cnn_k5 = CustomCNN(num_classes, kernel_size=5)
    train_cnn(cnn_k5, train_loader_img, val_loader_img, epochs=5)
    
    print("\n3. Transfer Learning (ResNet18)")
    try:
        weights = ResNet18_Weights.IMAGENET1K_V1
        resnet = resnet18(weights=weights)
    except:
        resnet = resnet18(pretrained=True)
        
    # Replace head
    resnet.fc = nn.Linear(resnet.fc.in_features, num_classes)
    
    # Fine-tune all layers (or freeze by setting requires_grad=False on parameters)
    train_cnn(resnet, train_loader_img, val_loader_img, epochs=5, lr=1e-4)


### Discussion Question 2 (CNN)
*   **Why are CNNs more parameter-efficient than MLPs for images?**
*   **How does Data Augmentation impact overfitting?**

*(Double-click to edit)*


# Part 3 & 4: RNN & Transformer on Jena Climate (Time Series)

**Task**: Forecast `T (degC)` (next time step or window).

**Models**:
1. **RNN/LSTM/GRU**: Experiments with Seq Length, Hidden Size, Architecture.
2. **Transformer**: `nn.TransformerEncoder` based model for comparison.


In [ ]:
def load_jena_data(root_path):
    csvs = list(root_path.glob("*.csv"))
    if not csvs: return None
    df = pd.read_csv(csvs[0])
    # Extract Temperature
    if "T (degC)" in df.columns:
        series = df["T (degC)"].values.astype(np.float32)
    else:
        series = df.iloc[:, 1].values.astype(np.float32) # Fallback
    return series

def create_sequences(data, seq_len):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i+seq_len])
        y.append(data[i+seq_len])
    return np.array(X), np.array(y)

if JENA_ROOT.exists():
    temperature_data = load_jena_data(JENA_ROOT)
    
    # Normalize
    mean_temp = temperature_data.mean()
    std_temp = temperature_data.std()
    data_norm = (temperature_data - mean_temp) / std_temp
    
    # Split (First 70% train, next 20% val, last 10% test)
    n = len(data_norm)
    train_data = data_norm[:int(0.7*n)]
    val_data = data_norm[int(0.7*n):int(0.9*n)]
    
    # Function to get loaders for specific seq_len
    def get_ts_loaders(seq_len=24, batch_size=64):
        X_train, y_train = create_sequences(train_data, seq_len)
        X_val, y_val = create_sequences(val_data, seq_len)
        
        train_ds = TensorDataset(torch.tensor(X_train).unsqueeze(-1), torch.tensor(y_train))
        val_ds = TensorDataset(torch.tensor(X_val).unsqueeze(-1), torch.tensor(y_val))
        
        return (
            DataLoader(train_ds, batch_size=batch_size, shuffle=True),
            DataLoader(val_ds, batch_size=batch_size, shuffle=False)
        )


In [ ]:
class RNNModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=1, rnn_type='lstm', dropout=0.0):
        super().__init__()
        if rnn_type == 'lstm':
            self.rnn = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        elif rnn_type == 'gru':
            self.rnn = nn.GRU(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        else:
            self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_size, 1)
        
    def forward(self, x):
        # x shape: (batch, seq_len, features)
        out, _ = self.rnn(x)
        # Take last time step
        out = out[:, -1, :]
        return self.fc(out).squeeze()

class TransformerTS(nn.Module):
    def __init__(self, input_size=1, d_model=64, nhead=4, num_layers=2):
        super().__init__()
        self.input_proj = nn.Linear(input_size, d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(d_model, 1)
        
    def forward(self, x):
        # x: (batch, seq, 1)
        x = self.input_proj(x) # (batch, seq, d_model)
        # Positional encoding could be added here
        out = self.transformer(x)
        # Average pooling or last token
        out = out[:, -1, :] 
        return self.fc(out).squeeze()

def train_ts(model, train_loader, val_loader, epochs=5, lr=1e-3):
    if train_loader is None: return
    model.to(DEVICE)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    for epoch in range(1, epochs+1):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            
        model.eval()
        val_mse = 0.0
        val_mae = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                preds = model(xb)
                val_mse += criterion(preds, yb).item()
                val_mae += nn.L1Loss()(preds, yb).item()
        
        print(f"Epoch {epoch} | Val MSE: {val_mse/len(val_loader):.4f} | Val MAE: {val_mae/len(val_loader):.4f}")


In [ ]:
if JENA_ROOT.exists():
    print("--- Sequence Modeling Experiments ---")
    
    train_loader_24, val_loader_24 = get_ts_loaders(seq_len=24)
    train_loader_72, val_loader_72 = get_ts_loaders(seq_len=72)

    print("\n1. LSTM (Seq=24)")
    model_lstm = RNNModel(rnn_type='lstm', hidden_size=64)
    train_ts(model_lstm, train_loader_24, val_loader_24, epochs=3)
    
    print("\n2. GRU (Seq=24) - Compare Speed/Convergence")
    model_gru = RNNModel(rnn_type='gru', hidden_size=64)
    train_ts(model_gru, train_loader_24, val_loader_24, epochs=3)
    
    print("\n3. LSTM (Seq=72) - Long Term Dependencies")
    model_lstm_long = RNNModel(rnn_type='lstm', hidden_size=64)
    train_ts(model_lstm_long, train_loader_72, val_loader_72, epochs=3)

    print("\n4. Transformer (Seq=24)")
    model_transformer = TransformerTS(input_size=1, d_model=32, nhead=2)
    train_ts(model_transformer, train_loader_24, val_loader_24, epochs=3)


### Discussion Question 3 (RNN & Transformers)
1. **Why are LSTMs/GRUs better than Vanilla RNNs?**
2. **Transformer vs. RNN: Advantages and Disadvantages?**
3. **What is Self-Attention?**

*(Double-click to edit)*


# Part 5: Research (Bonus)
## Which Machine Learning Models Are Actually Used in Industry?

*(Write your 1-page report here, citing sources like Kaggle State of Data Science, etc.)*

**Key Findings:**
...

**Future Predictions (5-10 Years):**
...